# DynamicAuc Tutorial

A complete walkthrough of the `dynamicauc` library: how to create scenarios,
what the available kernels model, how probes and delay samplers work together,
and how to interact with the Gym-compatible `AuctionEnv`.

## 1. Creating Scenarios

DynamicAuc offers three ways to build a `Scenario`, ordered from fastest to most flexible.

In [ ]:
import dynamicauc as dac
import numpy as np

# ── Method 1: Preset ──────────────────────────────────────────────────────────
# Pick a named scenario from the built-in registry. Best for reproducibility.
print(dac.describe_scenarios())

In [ ]:
# ── Method 2: Builder ─────────────────────────────────────────────────────────
# Compose primitives freely. Call order does not matter.
scenario_custom = (
    dac.ScenarioBuilder()
    .with_kernel("saturation", base_rate=0.7, saturation_scale=8.0, delay_rate=0.3)
    .with_auction("second-price")
    .with_competition("lognormal", mu=0.0, sigma=0.8)
    .with_arrival("cstpoisson", rate=1.5)
    .with_display("iid", dim=3)
    .with_exogenous("trivial")
    .with_horizon("fixed", value=30.0)
    .with_user_features("uniform", full_dim=3, observable_dim=3)
    .with_gamma("additive")
    .with_probe("poisson")        # Ogata thinning → 0..N delays
    .with_delay_sampler("multi")  # keep all sampled delays
    .with_max_history(10)
    .build()
)
print(f"Kernel  : {scenario_custom.kernel.__class__.__name__}")
print(f"Probe   : {scenario_custom.probe.__class__.__name__}")
print(f"Sampler : {scenario_custom.delay_sampler.__class__.__name__}")

In [ ]:
# ── Method 3: Hybrid ──────────────────────────────────────────────────────────
# Start from a preset, override selected parameters via underscore-path keys.
scenario_hybrid = dac.make(
    "mixed-second-price-exponential",
    horizon_value=50.0,          # override horizon.params.value
    kernel_kappa_base_rate=0.5,  # override kernel.params.kappa_base_rate
    max_history=5,
)
print(f"Horizon    : {scenario_hybrid.horizon_generator.__class__.__name__}")
print(f"kappa rate : {scenario_hybrid.kernel.kappa_base_rate}")
print(f"max_history: {scenario_hybrid.max_history}")

## 2. Kernels, Probes & Delay Samplers

This section explains the modelling primitives that control *how* a won auction generates conversions.

### 2.1 Reward Kernels

A `RewardKernel` defines **how a single display generates conversions over time**.
Every kernel decomposes the reward measure into an **immediate** part (at delay s=0) and a **delayed** part:

$$K(\cdot, ds) \;=\; \underbrace{\kappa(u, x, \text{ctx}, y_t)}_{\text{immediate}} \,\delta_0(ds)
\;+\; \underbrace{\lambda(u, x, \text{ctx}, y_{t+s}, t, s)}_{\text{delayed}} \,ds$$

The scalar **context** `ctx` is updated by a **gamma** function each time the advertiser wins an auction.
It encodes campaign history: saturation level, time of last display, etc.

| Kernel | `ctx` encodes | $\kappa$ | $\lambda(s)$ formula | Real-world scenario |
|---|---|---|---|---|
| `SaturationKernel` | Cumulative wins | 0 | `base_rate · exp(−ctx/scale) · delay_rate · exp(−delay_rate·s)` | Display-ad retargeting: repeated impressions fatigue the audience |
| `RecencyDisplayKernel` | Time of last display | 0 | `(1−exp(−α·τ)) · delay_rate · exp(−delay_rate·s)` where τ = gap since last display | Email/push: fresh contact is most effective |
| `CompetitorExtinctionKernel` | Unused (always 0) | 0 | `base_rate · delay_rate · exp(−delay_rate·s) · 𝟙[no competitor click]` | Search ads: a competitor click extinguishes the conversion window |
| `MixedKernel` | Delegates to base kernel | `kappa_base_rate > 0` | Delegates to base kernel | Rich-media/video: immediate click-through (κ>0) coexists with delayed view-through conversions |
| `SurvivalKernel` | Contextual scaling factor | 0 | `f(u,x,ctx) · α · exp(−α·s)` where f = clipped linear conversion probability | Single-purchase e-commerce: user buys at most once (Cox-style survival) |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from dynamicauc.primitives import MixedKernel, SaturationKernel

sns.set_theme(style='whitegrid')

s_vals = np.linspace(0, 15, 300)
t = 0.0
u, x = np.ones(3) * 0.5, np.ones(3) * 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# SaturationKernel: effect of cumulative exposure (context)
ax = axes[0]
for ctx, lbl in [(0, 'ctx=0 (fresh)'), (5, 'ctx=5'), (10, 'ctx=10 (saturated)')]:
    k = dac.SaturationKernel(base_rate=1.0, saturation_scale=10.0, delay_rate=0.4)
    ax.plot(s_vals, [k.lambda_(u, x, ctx, 0.0, t, s) for s in s_vals], label=lbl)
ax.set_title('SaturationKernel\n(more exposure → less response)', weight='bold')
ax.set_xlabel('Delay s'); ax.set_ylabel('λ(s)'); ax.legend(); ax.set_ylim(bottom=0)

# RecencyDisplayKernel: effect of gap since last display
ax = axes[1]
for tau, lbl in [(0.5, 'τ=0.5 (very recent)'), (3, 'τ=3'), (10, 'τ=10 (long ago)')]:
    k = dac.RecencyDisplayKernel(alpha=0.5, delay_rate=0.4)
    ctx = t - tau  # ctx stores the time of last display
    ax.plot(s_vals, [k.lambda_(u, x, ctx, 0.0, t, s) for s in s_vals], label=lbl)
ax.set_title('RecencyDisplayKernel\n(longer gap → stronger response)', weight='bold')
ax.set_xlabel('Delay s'); ax.legend(); ax.set_ylim(bottom=0)

# MixedKernel: varying κ (immediate) vs λ (delayed) balance
ax = axes[2]
base = SaturationKernel(base_rate=1.0, saturation_scale=10.0, delay_rate=0.4)
for kappa_rate, lbl in [(0.0, 'κ=0 (pure delayed)'), (0.3, 'κ=0.3'), (0.8, 'κ=0.8')]:
    k = MixedKernel(base, kappa_base_rate=kappa_rate)
    ax.plot(s_vals, [k.lambda_(u, x, 0.0, 0.0, t, s) for s in s_vals], label=f'{lbl} (λ)')
    if kappa_rate > 0:
        ax.axhline(kappa_rate, linestyle='--', alpha=0.4, label=f'κ={kappa_rate} (immediate)')
ax.set_title('MixedKernel\n(adds constant immediate κ on top of λ)', weight='bold')
ax.set_xlabel('Delay s'); ax.legend(fontsize=8); ax.set_ylim(bottom=0)

plt.suptitle('Delayed conversion intensity λ(s) across kernels (ctx=0)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

### 2.2 Architecture: ConversionProbe & DelaySampler

Once a display is won, two components translate the kernel's intensity into concrete delay samples.
The diagram below shows how `RewardKernel`, `ConversionProbe`, and `DelaySampler` compose:

![Delay sampler architecture](../images/delay_sampler_architecture.svg)

**Step 1 — ConversionProbe (purple):** wraps a kernel and samples raw candidate delays.

| Probe | Mechanism | Output | Use with |
|---|---|---|---|
| `PoissonProbe` | Ogata thinning on λ(s): proposes exponential inter-arrivals, accepts with probability λ(s)/λ_max | **0..N** delays | Any kernel with a well-defined λ |
| `SurvivalProbe` | Bernoulli(p) gate then Exp(α) draw | **0 or 1** delay | `SurvivalKernel` only |

**Step 2 — DelaySampler (teal):** decides how many delays to keep per display.

| Sampler | Method called | Semantics | Typical pairing |
|---|---|---|---|
| `SingleConversionSampler` | `probe.sample_one()` | At most **1** conversion per display | `SurvivalProbe` + `SurvivalKernel` |
| `MultiConversionSampler` | `probe.sample_all()` | **0..N** conversions per display | `PoissonProbe` + any Poisson-intensity kernel |

### 2.3 Built-in Preset Scenarios

Each preset bundles a kernel, probe, and sampler tuned to a specific modelling paradigm:

| Preset | Kernel | Probe | Sampler | Canonical use case |
|---|---|---|---|---|
| `paper-experiment-scenario` | `MixedKernel` (κ=0, saturation base) | Poisson | Multi | Paper baseline: pure delayed conversions, exponential campaign length |
| `saturation-second-price-uniform` | `SaturationKernel` | Poisson | Multi | Display retargeting: audience fatigue, many delayed clicks per impression |
| `recency-second-price-lognormal` | `RecencyDisplayKernel` | Poisson | Multi | Email/push: freshness of last touch drives response rate |
| `mixed-second-price-exponential` | `MixedKernel` (κ>0, saturation base) | Poisson | Multi | Rich-media/video: immediate click-through plus delayed view-through conversions |
| `survival-second-price-lognormal` | `SurvivalKernel` | Survival | Single | Single-purchase e-commerce: Cox-style model, user buys at most once |

In [ ]:
# Build every preset and inspect expected reward at campaign start
rng_ref = np.random.default_rng(42)
u_ref, x_ref = np.ones(3) * 0.5, np.ones(3) * 0.5

rows = []
for pid in dac.list_scenarios():
    sc = dac.make(pid)
    ctx0 = sc.gamma.initiate_context()
    horizon = min(sc.horizon_generator.sample(rng_ref), 30.0)
    er = sc.kernel.expected_reward(u_ref, x_ref, ctx0, lambda _: 0.0, t=0.0, horizon=horizon)
    rows.append({
        "preset": pid,
        "kernel": sc.kernel.__class__.__name__,
        "probe": sc.probe.__class__.__name__,
        "sampler": sc.delay_sampler.__class__.__name__,
        "E[reward | ctx=0]": round(er, 4),
    })

import pandas as pd
pd.set_option('display.max_colwidth', 35)
pd.DataFrame(rows).set_index('preset')

## 3. Setup and Imports
We load all components needed for the environment interaction demo below.

In [ ]:
import dynamicauc as dac
from dynamicauc.baseline.policies import FiPlaPolicy
import gymnasium as gym
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid')

## 4. Environment Interaction Loop
We use the `mixed-second-price-exponential` preset, which generates both immediate and delayed conversions. We run a single episode with `FiPlaPolicy` and record everything the agent observes.

In [ ]:
scenario = dac.make('mixed-second-price-exponential', horizon_value=50.0, kernel_base_kernel_delay_rate=0.5)
env = gym.make('DynamicAuction-v0', scenario=scenario)
policy = FiPlaPolicy()

obs, info = env.reset(seed=123)
done = False
history = []
cum_reward = 0.0

while not done:
    # Agent decides on a bid
    bid = policy.bid(obs)
    pre_step_obs = obs
    
    # Agent takes action
    obs, net_reward, done, truncated, step_info = env.step(bid)
    
    cum_reward += net_reward
    
    # Record state for visualization
    history.append({
        'step': pre_step_obs['step'],
        'time': pre_step_obs['t'].item(),
        'context': pre_step_obs['ctx'].item(),
        'expected_reward': pre_step_obs['expected_reward'].item(),
        'delayed_reward_observed': pre_step_obs['cumulative_reward_since_last_step'].item(),
        'conversion_times_since_last': [t.item() for t in pre_step_obs['conversion_times_since_last_step']],
        'bid': bid,
        'won': step_info['auction']['won'],
        'immediate_reward': net_reward,
        'cumulative_reward': cum_reward
    })

df = pd.DataFrame(history)
df.head(40)

## 5. Visualization: Auction Timeline
Let's map out when the agent bid, whether it won or lost, and when it received notifications of delayed conversions.

In [ ]:
def plot_auction_timeline(df):
    plt.figure(figsize=(14, 5))
    
    lost = df[~df['won']]
    won = df[df['won']]
    
    plt.scatter(lost['time'], lost['bid'], color='gray', alpha=0.5, label='Auction Lost (Bid)')
    plt.scatter(won['time'], won['bid'], color='green', s=80, label='Auction Won (Bid)', edgecolor='black')
    
    # Plot when delayed conversions were reported to the agent
    reported_times = []
    for _, row in df.iterrows():
        if row['delayed_reward_observed'] > 0:
            # Add one marker per conversion reported
            reported_times.extend([row['time']] * int(row['delayed_reward_observed']))
            
    if reported_times:
        plt.scatter(reported_times, [0] * len(reported_times), color='red', marker='*', s=200, label='Delayed Conversion Reported', zorder=5)
        
    plt.title('Auction Timeline & Actions', fontsize=14, weight='bold')
    plt.xlabel('Campaign Time $t$')
    plt.ylabel('Bid Value')
    plt.axhline(0, color='black', linewidth=1)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

plot_auction_timeline(df)

## 6. Visualization: Step-by-Step Reward Dynamics
Because of the delayed environment, the agent receives two types of feedback: immediate feedback for the current action, and delayed feedback from past actions that matured.

In [ ]:
def plot_reward_dynamics(df):
    plt.figure(figsize=(14, 5))
    
    plt.bar(df['time'], df['immediate_reward'], width=0.25, label='Immediate Net Reward', color='royalblue', alpha=0.8)
    
    # Stack delayed rewards on top of immediate positive rewards for visibility
    plt.bar(df['time'], df['delayed_reward_observed'], width=0.25, bottom=np.maximum(0, df['immediate_reward']), label='Delayed Reward Observed', color='darkorange', alpha=0.8)
    
    plt.title('Step-by-Step Reward Observation', fontsize=14, weight='bold')
    plt.xlabel('Campaign Time $t$')
    plt.ylabel('Reward Magnitude')
    plt.axhline(0, color='black', linewidth=1)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_reward_dynamics(df)

## 7. Visualization: Cumulative Metrics and Degradation
Finally, we can look at the overarching trend of the episode. As the agent wins more auctions, the cumulative reward goes up, but the `expected_reward` (value of future displays) goes down due to saturation.

In [ ]:
def plot_cumulative_metrics(df):
    fig, ax1 = plt.subplots(figsize=(14, 5))
    
    ax1.plot(df['time'], df['cumulative_reward'], 'b-', linewidth=2.5, label='Cumulative Net Reward')
    ax1.set_xlabel('Campaign Time $t$')
    ax1.set_ylabel('Cumulative Net Reward', color='b', weight='bold')
    ax1.tick_params(axis='y', labelcolor='b')
    
    ax2 = ax1.twinx()
    ax2.plot(df['time'], df['expected_reward'], 'r--', linewidth=2.5, label='Expected Reward (State)')
    ax2.set_ylabel('Expected Reward of Display $R_i$', color='r', weight='bold')
    ax2.tick_params(axis='y', labelcolor='r')
    
    plt.title('Evolution of Cumulative Reward and Expected Value', fontsize=14, weight='bold')
    fig.tight_layout()
    plt.show()

plot_cumulative_metrics(df)